# Step 0 — Data Preparation & Alignment

**Purpose:** Run iggnition alignment on all raw paired sequences, define naive/memory subsets,
compute per-sequence mutation summary statistics, apply QC filters, and save the master
analysis table used by all downstream steps.

**Output:** `processed/aligned_master.parquet`

**Design reference:** DESIGN.md §Step 0

In [ ]:
import polars as pl
import numpy as np
import iggnition
from pathlib import Path
import time

In [ ]:
DATA_DIR     = Path(".")
RAW_FILE     = DATA_DIR / "20260309_complete_database_with_lineages_and_subsets.parquet"
PROC_DIR     = DATA_DIR / "processed"
PROC_DIR.mkdir(exist_ok=True)

# Intermediate iggnition outputs (expensive to recompute)
MUTATED_FILE  = PROC_DIR / "iggnition_sequences.parquet"
GERMLINE_FILE = PROC_DIR / "iggnition_germlines.parquet"
MASTER_FILE   = PROC_DIR / "aligned_master.parquet"

## 0.1  Load raw data

In [ ]:
df = pl.read_parquet(RAW_FILE)
print(f"Total sequences: {df.height:,}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col!r:45s}  {str(df[col].dtype):<15}  "
          f"nulls={df[col].null_count():,}  "
          f"example={df[col][0]}")

## 0.2  Naive / memory subset definitions

Three operationally distinct definitions of naïve B cells:

| Label | Definition |
|---|---|
| `naive_bio` | Biological label: `subset == 'naive'` (CD27⁻ MACS separation) |
| `naive_comp` | Computational: IgM + V-gene identity ≥ 99% |
| `naive_strict` | Intersection of both (most conservative) |

All three are retained in the master table for comparative analyses.

In [ ]:
df = df.with_columns([
    # Method 1: biological label from wetlab CD27- MACS separation
    (pl.col('subset') == 'naive').alias('naive_bio'),

    # Method 2: computational — IgM + < 1% SHM in V gene of heavy chain
    ((pl.col('c_gene:0') == 'IGHM') & (pl.col('v_identity:0') >= 0.99)).alias('naive_comp'),
])

df = df.with_columns([
    # Method 3: intersection of both
    (pl.col('naive_bio') & pl.col('naive_comp')).alias('naive_strict'),
])

N = df.height
for label in ('naive_bio', 'naive_comp', 'naive_strict'):
    n = df.filter(pl.col(label)).height
    print(f"{label:20s}: {n:>9,}  ({100*n/N:.1f}%)")

# Concordance: how often do bio and comp agree?
agree = df.filter(pl.col('naive_bio') == pl.col('naive_comp')).height
print(f"\nBio == Comp concordance: {agree:,} / {N:,} ({100*agree/N:.2f}%)")

bio_not_comp = df.filter(pl.col('naive_bio') & ~pl.col('naive_comp')).height
comp_not_bio = df.filter(pl.col('naive_comp') & ~pl.col('naive_bio')).height
print(f"Bio-naïve but NOT comp-naïve: {bio_not_comp:,}  (IgM-switched or mutated CD27- cells)")
print(f"Comp-naïve but NOT bio-naïve: {comp_not_bio:,}  (unmutated IgM in memory fraction)")

## 0.3  iggnition alignment

Run iggnition on **all** sequences (naïve + memory) to produce IMGT/Aho-numbered nucleotide
alignments in wide format.  
Results are cached to parquet — skip if already computed.

In [ ]:
if MUTATED_FILE.exists():
    print(f"Loading cached sequences from {MUTATED_FILE}")
    mutated = pl.read_parquet(MUTATED_FILE)
else:
    print("Running iggnition on sequences...")
    t0 = time.time()
    mutated, _ = iggnition.run(df, name_col="name", wide=True, verbose=True, human_readable=False)
    print(f"Done in {time.time()-t0:.0f}s — shape: {mutated.shape}")
    mutated.write_parquet(MUTATED_FILE)
    print(f"Saved → {MUTATED_FILE}")

mutated.head(3)

In [ ]:
if GERMLINE_FILE.exists():
    print(f"Loading cached germlines from {GERMLINE_FILE}")
    germline = pl.read_parquet(GERMLINE_FILE)
else:
    print("Running iggnition on germlines...")
    t0 = time.time()
    germline, _ = iggnition.run(
        df, paired=True,
        nt_col_heavy="germline:0",  aa_col_heavy="germline_aa:0",
        nt_col_light="germline:1",  aa_col_light="germline_aa:1",
        name_col="name", wide=True, verbose=True, human_readable=False,
    )
    print(f"Done in {time.time()-t0:.0f}s — shape: {germline.shape}")
    germline.write_parquet(GERMLINE_FILE)
    print(f"Saved → {GERMLINE_FILE}")

germline.head(3)

## 0.4  Mutation matrix

Per-sequence, per-nucleotide-position mutation indicator: `sequence_nt − germline_nt`.
- `0`   = no change
- non-zero = nucleotide mutation (value encodes direction via ASCII difference)
- `null`   = position absent in alignment

In [ ]:
KEY = "seq_name"
pos_cols = [c for c in mutated.columns if c != KEY]

mutations = (
    mutated
    .join(germline, on=KEY, how="inner", suffix="_germ")
    .with_columns([
        (pl.col(c) - pl.col(f"{c}_germ")).alias(c)
        for c in pos_cols
    ])
    .select([KEY] + pos_cols)
)

print(f"Mutation matrix: {mutations.shape}")
print(f"Sequences with alignment: {mutations.height:,} "
      f"(dropped {df.height - mutations.height:,} without germline match)")

## 0.5  Region maps (Aho nucleotide coordinates)

Aho amino acid boundaries (from iggnition `regions.py`, branch `map`):

| Region | Heavy | Light |
|--------|-------|-------|
| FR1    | 1–25  | 1–25  |
| CDR1   | 26–38 | 26–38 |
| FR2    | 39–49 | 39–49 |
| CDR2   | 50–64 | 50–66 |
| FR3    | 65–108| 67–108|
| CDR3   | 109–137| 109–138|
| FR4    | 138–149| 139–148|

Nucleotide column formula: Aho AA position `k` → nt columns `{chain}{(k-1)*3+1}` … `{chain}{k*3}`

In [ ]:
def aho_to_nt_cols(chain: str, aho_start: int, aho_end: int) -> list[str]:
    """Return nt column names for Aho AA positions aho_start..aho_end (inclusive)."""
    return [f"{chain}{i}" for i in range((aho_start - 1) * 3 + 1, aho_end * 3 + 1)]


# Heavy chain region columns (nucleotide-level)
H_REGIONS = {
    'FR1' : aho_to_nt_cols('H',   1,  25),
    'CDR1': aho_to_nt_cols('H',  26,  38),
    'FR2' : aho_to_nt_cols('H',  39,  49),
    'CDR2': aho_to_nt_cols('H',  50,  64),
    'FR3' : aho_to_nt_cols('H',  65, 108),
    'CDR3': aho_to_nt_cols('H', 109, 137),
    'FR4' : aho_to_nt_cols('H', 138, 149),
}

# Light chain region columns (nucleotide-level; K and L share boundaries here)
L_REGIONS = {
    'FR1' : aho_to_nt_cols('L',   1,  25),
    'CDR1': aho_to_nt_cols('L',  26,  38),
    'FR2' : aho_to_nt_cols('L',  39,  49),
    'CDR2': aho_to_nt_cols('L',  50,  66),
    'FR3' : aho_to_nt_cols('L',  67, 108),
    'CDR3': aho_to_nt_cols('L', 109, 138),
    'FR4' : aho_to_nt_cols('L', 139, 148),
}

# Convenience groupings
H_CDR_COLS = H_REGIONS['CDR1'] + H_REGIONS['CDR2'] + H_REGIONS['CDR3']
H_FWR_COLS = H_REGIONS['FR1']  + H_REGIONS['FR2']  + H_REGIONS['FR3'] + H_REGIONS['FR4']
L_CDR_COLS = L_REGIONS['CDR1'] + L_REGIONS['CDR2'] + L_REGIONS['CDR3']
L_FWR_COLS = L_REGIONS['FR1']  + L_REGIONS['FR2']  + L_REGIONS['FR3'] + L_REGIONS['FR4']

# Filter to columns actually present in the data
present = set(mutations.columns)
H_CDR_COLS = [c for c in H_CDR_COLS if c in present]
H_FWR_COLS = [c for c in H_FWR_COLS if c in present]
L_CDR_COLS = [c for c in L_CDR_COLS if c in present]
L_FWR_COLS = [c for c in L_FWR_COLS if c in present]

print(f"H CDR cols: {len(H_CDR_COLS)} nt  ({len(H_CDR_COLS)//3} codons)")
print(f"H FWR cols: {len(H_FWR_COLS)} nt  ({len(H_FWR_COLS)//3} codons)")
print(f"L CDR cols: {len(L_CDR_COLS)} nt  ({len(L_CDR_COLS)//3} codons)")
print(f"L FWR cols: {len(L_FWR_COLS)} nt  ({len(L_FWR_COLS)//3} codons)")

## 0.6  Mutation summary counts

### 0.6a  Codon-level mutation counts by chain and region

A codon is counted as mutated if **any** of its 3 nucleotide positions differs from germline.
This gives counts in amino acid units (comparable across sequences regardless of CDR3 length).

In [ ]:
def codon_mut_count_expr(cols: list[str]) -> pl.Expr:
    """
    Count the number of mutated codons in a set of nt columns.
    A codon (3 consecutive cols) is mutated if any of its 3 positions != 0.
    Null positions are treated as unmutated.
    """
    assert len(cols) % 3 == 0, "Column list must be divisible into triplets"
    codon_flags = []
    for i in range(0, len(cols), 3):
        c1, c2, c3 = cols[i], cols[i+1], cols[i+2]
        codon_mutated = (
            (pl.col(c1).fill_null(0) != 0) |
            (pl.col(c2).fill_null(0) != 0) |
            (pl.col(c3).fill_null(0) != 0)
        ).cast(pl.UInt16)
        codon_flags.append(codon_mutated)
    return pl.sum_horizontal(codon_flags)


summary = mutations.select([
    pl.col(KEY),
    codon_mut_count_expr(H_CDR_COLS + H_FWR_COLS).alias('n_mut_H'),
    codon_mut_count_expr(L_CDR_COLS + L_FWR_COLS).alias('n_mut_L'),
    codon_mut_count_expr(H_CDR_COLS).alias('n_mut_CDR_H'),
    codon_mut_count_expr(H_FWR_COLS).alias('n_mut_FWR_H'),
    codon_mut_count_expr(L_CDR_COLS).alias('n_mut_CDR_L'),
    codon_mut_count_expr(L_FWR_COLS).alias('n_mut_FWR_L'),
])

summary = summary.with_columns(
    (pl.col('n_mut_H') + pl.col('n_mut_L')).alias('n_mut_total')
)

print(summary.describe())

### 0.6b  Replacement (R) vs Silent (S) classification

For each mutated codon: translate germline and sequence codons using the standard genetic code.
- **R**: amino acid changed  
- **S**: nucleotide changed but amino acid preserved  

Computed separately for CDR and framework regions (per chain).

In [ ]:
# Standard genetic code: codon string → amino acid char
GENETIC_CODE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

# Integer-keyed lookup: a*65536 + b*256 + c (ASCII) → AA ordinal
# Returns 0 for unknown/null codons
AA_LOOKUP = {0: 0}  # null sentinel
for codon, aa in GENETIC_CODE.items():
    k = ord(codon[0]) * 65536 + ord(codon[1]) * 256 + ord(codon[2])
    AA_LOOKUP[k] = ord(aa)
_aa_lookup_vec = np.vectorize(lambda k: AA_LOOKUP.get(int(k), 0), otypes=[np.int32])


def translate_codon_array(nt: np.ndarray) -> np.ndarray:
    """Translate (N, 3) uint8/int32 nt array → (N,) int32 AA ordinals. 0 = null/invalid."""
    keys = nt[:, 0].astype(np.int32) * 65536 + \
           nt[:, 1].astype(np.int32) * 256  + \
           nt[:, 2].astype(np.int32)
    return _aa_lookup_vec(keys)


def count_rs(germ_nt: np.ndarray, seq_nt: np.ndarray,
             codon_col_indices: list[int]) -> tuple[np.ndarray, np.ndarray]:
    """
    Count replacement (R) and silent (S) codon mutations.

    Parameters
    ----------
    germ_nt, seq_nt : (N, L) int32 arrays, nucleotide ASCII values; 0 = null/gap
    codon_col_indices : 0-based column indices for the FIRST nt of each codon

    Returns
    -------
    n_R, n_S : (N,) int16 arrays
    """
    N = germ_nt.shape[0]
    n_R = np.zeros(N, dtype=np.int16)
    n_S = np.zeros(N, dtype=np.int16)

    for start in codon_col_indices:
        g = germ_nt[:, start:start+3]   # (N, 3)
        s = seq_nt[:,  start:start+3]   # (N, 3)

        # Valid = both germline and sequence have non-null nt at all 3 positions
        valid = np.all(g > 0, axis=1) & np.all(s > 0, axis=1)
        has_mut = valid & np.any(g != s, axis=1)

        if not np.any(has_mut):
            continue

        g_aa = translate_codon_array(g)   # (N,)
        s_aa = translate_codon_array(s)   # (N,)
        valid_aa = (g_aa > 0) & (s_aa > 0)

        n_R += (has_mut & (g_aa != s_aa) & valid_aa).astype(np.int16)
        n_S += (has_mut & (g_aa == s_aa) & valid_aa).astype(np.int16)

    return n_R, n_S

print("Functions defined.")

In [ ]:
# Align mutated and germline to the same rows (inner join already done in mutations)
aligned_names = mutations.select(KEY).to_series().to_list()

# Extract numpy arrays for H and L chains (fill null → 0)
H_COLS_ALL = [c for c in pos_cols if c.startswith('H') and c in present]
L_COLS_ALL = [c for c in pos_cols if c.startswith('L') and c in present]

# Sort column names by nt position number
H_COLS_ALL = sorted(H_COLS_ALL, key=lambda c: int(c[1:]))
L_COLS_ALL = sorted(L_COLS_ALL, key=lambda c: int(c[1:]))

# We need the original nt values (not differences) for translation
# Align mutated and germline to the joined set
mut_aligned  = mutated.filter(pl.col(KEY).is_in(aligned_names))
germ_aligned = germline.filter(pl.col(KEY).is_in(aligned_names))

# Sort both by KEY to ensure row alignment
mut_aligned  = mut_aligned.sort(KEY)
germ_aligned = germ_aligned.sort(KEY)
mutations_sorted = mutations.sort(KEY)

print("Extracting H chain numpy arrays...")
germ_H = germ_aligned.select(H_COLS_ALL).fill_null(0).to_numpy().astype(np.int32)
seq_H  = mut_aligned.select(H_COLS_ALL).fill_null(0).to_numpy().astype(np.int32)

print("Extracting L chain numpy arrays...")
germ_L = germ_aligned.select(L_COLS_ALL).fill_null(0).to_numpy().astype(np.int32)
seq_L  = mut_aligned.select(L_COLS_ALL).fill_null(0).to_numpy().astype(np.int32)

print(f"H arrays: {germ_H.shape}   L arrays: {germ_L.shape}")

In [ ]:
def region_codon_starts(region_cols: list[str], all_cols: list[str]) -> list[int]:
    """Return 0-based column indices for the first nt of each codon in region_cols."""
    col_to_idx = {c: i for i, c in enumerate(all_cols)}
    starts = []
    for i in range(0, len(region_cols), 3):
        c = region_cols[i]
        if c in col_to_idx:
            starts.append(col_to_idx[c])
    return starts


# Filter region columns to those present in data
H_CDR = [c for c in H_CDR_COLS if c in set(H_COLS_ALL)]
H_FWR = [c for c in H_FWR_COLS if c in set(H_COLS_ALL)]
L_CDR = [c for c in L_CDR_COLS if c in set(L_COLS_ALL)]
L_FWR = [c for c in L_FWR_COLS if c in set(L_COLS_ALL)]

print("Computing R/S for H CDR...")
t0 = time.time()
R_CDR_H, S_CDR_H = count_rs(germ_H, seq_H, region_codon_starts(H_CDR, H_COLS_ALL))
print(f"  done in {time.time()-t0:.1f}s")

print("Computing R/S for H FWR...")
t0 = time.time()
R_FWR_H, S_FWR_H = count_rs(germ_H, seq_H, region_codon_starts(H_FWR, H_COLS_ALL))
print(f"  done in {time.time()-t0:.1f}s")

print("Computing R/S for L CDR...")
t0 = time.time()
R_CDR_L, S_CDR_L = count_rs(germ_L, seq_L, region_codon_starts(L_CDR, L_COLS_ALL))
print(f"  done in {time.time()-t0:.1f}s")

print("Computing R/S for L FWR...")
t0 = time.time()
R_FWR_L, S_FWR_L = count_rs(germ_L, seq_L, region_codon_starts(L_FWR, L_COLS_ALL))
print(f"  done in {time.time()-t0:.1f}s")

rs_df = pl.DataFrame({
    KEY:               mutations_sorted[KEY].to_list(),
    'n_R_CDR_H':       R_CDR_H,
    'n_S_CDR_H':       S_CDR_H,
    'n_R_FWR_H':       R_FWR_H,
    'n_S_FWR_H':       S_FWR_H,
    'n_R_CDR_L':       R_CDR_L,
    'n_S_CDR_L':       S_CDR_L,
    'n_R_FWR_L':       R_FWR_L,
    'n_S_FWR_L':       S_FWR_L,
})

rs_df = rs_df.with_columns([
    (pl.col('n_R_CDR_H') + pl.col('n_R_CDR_L')).alias('n_R_CDR'),
    (pl.col('n_S_CDR_H') + pl.col('n_S_CDR_L')).alias('n_S_CDR'),
    (pl.col('n_R_FWR_H') + pl.col('n_R_FWR_L')).alias('n_R_FWR'),
    (pl.col('n_S_FWR_H') + pl.col('n_S_FWR_L')).alias('n_S_FWR'),
])

print("\nR/S summary:")
print(rs_df.select(['n_R_CDR','n_S_CDR','n_R_FWR','n_S_FWR']).describe())

## 0.7  CDRH3 length

CDRH3 amino acid length = number of non-null nt positions in the CDR3-H Aho range (109–137) ÷ 3.  
Null positions in CDR3 correspond to gap-filled alignment slots (shorter loops).

In [ ]:
cdr3_H_cols = [c for c in H_REGIONS['CDR3'] if c in present]

cdrh3_len = (
    mut_aligned
    .select([KEY] + cdr3_H_cols)
    .with_columns(
        # Count non-null nt positions in CDR3, divide by 3 → AA length
        (pl.sum_horizontal([pl.col(c).is_not_null().cast(pl.UInt16) for c in cdr3_H_cols]) // 3)
        .alias('cdrh3_length')
    )
    .select([KEY, 'cdrh3_length'])
)

print(cdrh3_len.select('cdrh3_length').describe())

## 0.8  Quality control

Remove sequences that would corrupt downstream statistics:
1. Stop codons in V region (detected as `*` in translation)
2. Missing heavy **or** light chain alignment (null in H1 or L1)
3. Ambiguous germline assignment (if confidence score available)

Ambiguous-nucleotide sequences (N content) should have been removed by iggnition during alignment;
we verify below.

In [ ]:
# Flag stop codons in V-region of heavy chain (FR1–FR3, Aho 1–108)
H_V_COLS = [c for c in H_REGIONS['FR1'] + H_REGIONS['CDR1'] + H_REGIONS['FR2'] +
                        H_REGIONS['CDR2'] + H_REGIONS['FR3'] if c in present]
H_V_codon_starts = list(range(0, len(H_V_COLS) - 2, 3))

print("Detecting stop codons in heavy chain V region...")
STOP_AA = ord('*')
germ_H_V = germ_H[:, :len(H_V_COLS)]
seq_H_V  = seq_H[:,  :len(H_V_COLS)]

has_stop = np.zeros(germ_H.shape[0], dtype=bool)
for start in H_V_codon_starts:
    s_aa = translate_codon_array(seq_H_V[:, start:start+3])
    has_stop |= (s_aa == STOP_AA)

n_stop = has_stop.sum()
print(f"  Sequences with stop codon in V-H region: {n_stop:,}")

stop_names = set(
    pl.Series(mutations_sorted[KEY].to_list()).filter(pl.Series(has_stop)).to_list()
)

# Flag missing chain alignment (null H1 or null L1)
missing_H = mut_aligned.filter(pl.col('H1').is_null()).select(KEY).to_series().to_list()
missing_L = mut_aligned.filter(pl.col('L1').is_null()).select(KEY).to_series().to_list()
print(f"  Missing H alignment: {len(missing_H):,}")
print(f"  Missing L alignment: {len(missing_L):,}")

fail_qc = stop_names | set(missing_H) | set(missing_L)
print(f"\nTotal failing QC: {len(fail_qc):,}  "
      f"(retaining {mutations.height - len(fail_qc):,} sequences)")

## 0.9  Assemble master table

In [ ]:
# Metadata from original df, keyed by name → seq_name
meta = df.rename({'name': KEY})

master = (
    mutations_sorted
    .select(KEY)                                        # start from aligned set
    .join(meta,        on=KEY, how='left')              # add all metadata
    .join(summary,     on=KEY, how='left')              # mutation counts
    .join(rs_df,       on=KEY, how='left')              # R/S classification
    .join(cdrh3_len,   on=KEY, how='left')              # CDRH3 length
    .filter(~pl.col(KEY).is_in(list(fail_qc)))          # QC filter
)

print(f"Master table: {master.shape}")
print(master.head(3))

## 0.10  Validation sanity checks

Expected patterns:
- Naïve sequences → mutation counts near zero
- Memory sequences → VH mutation peak at 5–15 codons
- CDR enrichment > FWR on average in memory (positive selection signal)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) Mutation count distribution by naive definition
ax = axes[0]
for label, mask in [('naive_bio', 'naive_bio'), ('naive_comp', 'naive_comp'),
                    ('naive_strict', 'naive_strict'), ('memory', None)]:
    subset = master.filter(pl.col(mask)) if mask else \
             master.filter(~pl.col('naive_bio') & ~pl.col('naive_comp'))
    vals = subset['n_mut_H'].drop_nulls().to_numpy()
    ax.hist(vals, bins=50, alpha=0.5, label=f"{label} (n={len(vals):,})", density=True)
ax.set_xlabel('VH codon mutations')
ax.set_ylabel('Density')
ax.set_title('Mutation load by subset')
ax.legend(fontsize=7)
ax.set_xlim(0, 50)

# 2) CDR vs FWR mutation ratio in memory sequences
ax = axes[1]
mem = master.filter(~pl.col('naive_bio')).filter(pl.col('n_mut_H') > 0)
cdr_frac = (mem['n_mut_CDR_H'] / mem['n_mut_H'].cast(pl.Float32)).drop_nulls().to_numpy()
ax.hist(cdr_frac, bins=50, color='steelblue', edgecolor='none')
ax.axvline(len(H_CDR_COLS)/len(H_CDR_COLS + H_FWR_COLS), color='red',
           linestyle='--', label='Neutral expectation')
ax.set_xlabel('Fraction of VH mutations in CDR')
ax.set_ylabel('Count')
ax.set_title('CDR mutation enrichment (memory)')
ax.legend()

# 3) CDRH3 length distribution
ax = axes[2]
ax.hist(master['cdrh3_length'].drop_nulls().to_numpy(), bins=30,
        color='coral', edgecolor='none')
ax.set_xlabel('CDRH3 length (aa)')
ax.set_ylabel('Count')
ax.set_title('CDRH3 length distribution')

plt.tight_layout()
plt.savefig(PROC_DIR / 'step0_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved validation figure.")

## 0.11  Save

In [ ]:
master.write_parquet(MASTER_FILE)
print(f"Saved master table → {MASTER_FILE}")
print(f"Shape: {master.shape}")
print(f"\nColumn list:")
for c in master.columns:
    print(f"  {c}")